# HW4 – Object Detection: Loss Functions, Evaluation Metrics & Implementation

Citations:

Code Formatted in Claude Code

Understanding and Paper Expalination in Gemini: https://gemini.google.com/share/8b7d5c4575c9

I watched this youtube video and relied heavily on it for understanding: https://www.youtube.com/watch?v=c_nEue9itwg&t=1046s

## Part 1: Paper Review – Loss Function & Evaluation Metrics (40 points)

**Selected Model: SSD (Single Shot MultiBox Detector)**

---

### 1. Architecture Summary

SSD is a single-stage object detector that eliminates the need for a separate region proposal network, enabling real-time inference without sacrificing significant accuracy.

**Backbone:**  
SSD uses a base classification network — in the original paper, VGG-16 — truncated after the `conv5_3` layer. The fully connected layers (`fc6`, `fc7`) are converted to convolutional layers, forming the core feature extractor. 

**Multi-Scale Feature Maps (Detection Head):**  
The key architectural innovation of SSD is its use of multiple feature maps at different scales to detect objects of varying sizes. After the base network, several auxiliary convolutional layers of progressively decreasing spatial resolution are appended (e.g., `conv6` through `conv11` in the original design). Detections are made from six feature maps in total:

| Layer | Feature Map Size | Receptive Field |
|-------|-----------------|-----------------|
| `conv4_3` | 38 × 38 | Small objects |
| `fc7` | 19 × 19 | Medium objects |
| `conv6_2` | 10 × 10 | Medium objects |
| `conv7_2` | 5 × 5 | Larger objects |
| `conv8_2` | 3 × 3 | Large objects |
| `conv9_2` | 1 × 1 | Largest objects |

**Default Boxes (Anchors):**  
At each location on each feature map, SSD tiles a set of *default boxes* (analogous to anchors in Faster R-CNN) with predefined aspect ratios (e.g., 1:1, 2:1, 1:2, 3:1, 1:3) and scales. The scale of default boxes grows with the depth of the feature map, so shallow layers handle small objects and deeper layers handle large ones.

**Bounding Box & Class Predictions:**  
For each default box, the network predicts:
- **4 offsets** (Δcx, Δcy, Δw, Δh) relative to the default box to regress the final bounding box.
- **C class scores** (one per class, including a background class) via a softmax or sigmoid over `C + 1` outputs.

These predictions are produced by small 3×3 convolutional filters applied independently to each feature map, keeping the parameter count manageable. The total number of predictions across all feature map locations and all default boxes is ~8,732 per image in the 300×300 input variant (SSD300).


---

### 3. Loss Function

#### Full Loss Formula

SSD minimizes a weighted sum of a localization loss and a confidence (classification) loss over all matched default boxes:

$$L(x,\, c,\, l,\, g) = \frac{1}{N} \Bigl( L_{\text{conf}}(x,\, c) \;+\; \alpha \cdot L_{\text{loc}}(x,\, l,\, g) \Bigr)$$

where $N$ is the number of matched (positive) default boxes. If $N = 0$, the loss is set to zero.

---

#### Component 1 – Localization Loss $L_{\text{loc}}$

$$L_{\text{loc}}(x,\, l,\, g) = \sum_{i \in \text{Pos}}^{N} \sum_{m \in \{cx,\, cy,\, w,\, h\}} x_{ij}^{k} \;\cdot\; \text{smooth}_{L_1}\!\bigl(l_i^m - \hat{g}_j^m\bigr)$$

The encoded ground-truth offsets are:

$$\hat{g}_j^{cx} = \frac{g_j^{cx} - d_i^{cx}}{d_i^{w}}, \qquad \hat{g}_j^{cy} = \frac{g_j^{cy} - d_i^{cy}}{d_i^{h}}, \qquad \hat{g}_j^{w} = \log\!\frac{g_j^{w}}{d_i^{w}}, \qquad \hat{g}_j^{h} = \log\!\frac{g_j^{h}}{d_i^{h}}$$

and the Smooth-$L_1$ function is:

$$\text{smooth}_{L_1}(x) = \begin{cases} 0.5\,x^2 & \text{if } |x| < 1 \\ |x| - 0.5 & \text{otherwise} \end{cases}$$

**What it does:** Measures how accurately the network regresses each predicted box's center coordinates and dimensions relative to the matched default box. The sum runs only over *positive* (matched) default boxes — there is no localization penalty for background boxes.

**Loss type:** Smooth $L_1$ (Huber loss). Compared to plain $L_2$, it is less sensitive to large offset errors early in training, preventing exploding gradients.

---

#### Component 2 – Confidence Loss $L_{\text{conf}}$

$$L_{\text{conf}}(x,\, c) = -\sum_{i \in \text{Pos}}^{N} x_{ij}^{p}\,\log\!\bigl(\hat{c}_i^{p}\bigr) \;-\; \sum_{i \in \text{Neg}} \log\!\bigl(\hat{c}_i^{0}\bigr)$$

where the predicted class probability is a softmax over all $C+1$ classes (including background class 0):

$$\hat{c}_i^{p} = \frac{\exp(c_i^{p})}{\displaystyle\sum_{p'} \exp(c_i^{p'})}$$

**What it does:** Penalizes wrong class assignments for every default box, positive boxes must predict the correct object class, while negative boxes must predict *background*. Because positive boxes are vastly outnumbered by negatives, SSD applies **Hard Negative Mining**: after sorting negatives by highest confidence loss (i.e., the ones the model is most confidently wrong about), only the top negatives are kept so that the ratio of negatives to positives is at most **3:1**. This keeps the training set balanced and avoids the loss being dominated by easy background examples.

**Loss type:** Multi-class cross-entropy (softmax log-loss).

---

#### Weighting Hyperparameter $\alpha$

The scalar $\alpha$ balances the relative contribution of localization versus classification error in the total loss. In the SSD paper it is set to **$\alpha = 1$**, determined by cross-validation. A value of 1 means both losses contribute equally (after the $1/N$ normalization). If $\alpha$ were increased, the model would be penalized more heavily for imprecise box regression; if decreased, classification accuracy would dominate training. Unlike YOLO's multiple $\lambda$ terms (separate weights for coordinate, objectness, and no-object losses) or RetinaNet's focal-loss parameters ($\alpha_{\text{FL}},\, \gamma$), SSD uses just this single balancing scalar, keeping the loss design simple.

---

### 4. Evaluation Metrics

---

#### Intersection over Union (IoU)

$$\text{IoU}(B_{\text{pred}},\, B_{\text{gt}}) = \frac{|B_{\text{pred}} \cap B_{\text{gt}}|}{|B_{\text{pred}} \cup B_{\text{gt}}|}$$

where $B_{\text{pred}}$ is the predicted bounding box and $B_{\text{gt}}$ is the ground-truth bounding box. The numerator is the area of their overlap; the denominator is the area of their union.

**Why it is used:** IoU gives a single normalized score in $[0, 1]$ that captures both localization accuracy and overlap quality simultaneously — a prediction that is offset, too small, or too large all reduce the score. It is used as a threshold gate: a predicted box is counted as a *true positive* only if its IoU with a ground-truth box exceeds a chosen threshold (e.g., 0.5), otherwise it is a *false positive*. This makes the metric sensitive to precise box placement, not just whether the object was found somewhere nearby.

---

#### Average Precision (AP) and mean Average Precision (mAP)

**Building the Precision–Recall curve:**

1. For a given class, collect all predicted boxes across the entire test set, each with a confidence score.
2. Sort predictions by confidence score in descending order.
3. Step through the ranked list: for each prediction, check whether it is a true positive (IoU ≥ threshold with an unmatched ground-truth box of that class) or a false positive.
4. At each step $k$, compute:

$$\text{Precision}(k) = \frac{TP_k}{TP_k + FP_k}, \qquad \text{Recall}(k) = \frac{TP_k}{N_{\text{gt}}}$$

where $N_{\text{gt}}$ is the total number of ground-truth instances of that class. Plotting these pairs traces the precision–recall (PR) curve.

**Average Precision (AP)** is the area under the (interpolated) PR curve for a single class. The VOC 2010+ standard uses **all-point interpolation**:

$$AP = \int_0^1 p_{\text{interp}}(r)\, dr \approx \sum_{r \in \{0,\, 0.01,\, \ldots,\, 1.0\}} p_{\text{interp}}(r)$$

where $p_{\text{interp}}(r) = \max_{\tilde{r} \geq r} p(\tilde{r})$ (the maximum precision at any recall $\geq r$). This smoothing removes the zigzag of the raw curve.

**mean Average Precision (mAP)** is simply the mean of AP over all $C$ object classes:

$$mAP = \frac{1}{C} \sum_{c=1}^{C} AP_c$$

---

#### IoU Thresholds Used in the SSD Paper

The SSD paper reports results on two benchmarks with different threshold conventions:

| Benchmark | IoU Threshold | Notation |
|---|---|---|
| **PASCAL VOC 2007 / 2012** | 0.50 | mAP@0.5 |
| **MS COCO** | 0.50 : 0.05 : 0.95 (average over 10 thresholds) | mAP@[0.5:0.95] |
| MS COCO (loose) | 0.50 only | AP₅₀ |
| MS COCO (strict) | 0.75 only | AP₇₅ |

PASCAL VOC mAP@0.5 is the primary metric cited in the paper (e.g., SSD300 achieves **74.3 mAP** on VOC 2007 test). The COCO metric mAP@[0.5:0.95] is stricter because it rewards finer localization at higher thresholds; SSD300 achieves **23.2 AP** on COCO under this criterion.

---

#### Additional Metrics

**Frames Per Second (FPS):** The SSD paper places heavy emphasis on inference speed alongside accuracy. SSD300 runs at **~59 FPS** on a Titan X GPU, compared to ~7 FPS for Faster R-CNN at similar accuracy. FPS is reported alongside mAP throughout the paper, framing SSD as a speed–accuracy trade-off rather than a pure accuracy maximizer. This makes it one of the few detection papers of its era to treat latency as a first-class evaluation criterion alongside standard detection accuracy.

---

## Part 2: Running the Model & Evaluation (60 points)

### Task 2A: Run the Pretrained Model

**Dataset:** COCO 2017 validation split (300-image subset)  
**Model:** `ssd300_vgg16` from `torchvision.models.detection`  
**Weights:** `SSD300_VGG16_Weights.COCO_V1` (pretrained on COCO train2017)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "torchvision", "pycocotools", "requests", "tqdm"])

In [ ]:
import json, random, zipfile, requests
from pathlib import Path
from tqdm import tqdm

IMG_DIR = Path("coco_val_subset/images")
ANN_FILE = Path("coco_val_subset/instances_val2017.json")
IMG_DIR.mkdir(parents=True, exist_ok=True)

# download + extract annotations once
if not ANN_FILE.exists():
    zip_path = ANN_FILE.with_suffix(".zip")
    data = requests.get("http://images.cocodataset.org/annotations/annotations_trainval2017.zip").content
    zip_path.write_bytes(data)
    with zipfile.ZipFile(zip_path) as z:
        src = "annotations/instances_val2017.json"
        ANN_FILE.write_bytes(z.read(src))
    zip_path.unlink()

coco = json.load(open(ANN_FILE))
random.seed(42)
sampled = random.sample(coco["images"], 300)

# download images
for img in tqdm(sampled, desc="images"):
    dest = IMG_DIR / img["file_name"]
    if not dest.exists():
        dest.write_bytes(requests.get("http://images.cocodataset.org/val2017/" + img["file_name"]).content)

print(f"Ready: {len(list(IMG_DIR.iterdir()))} images")

In [ ]:
import torch
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights

WEIGHTS = SSD300_VGG16_Weights.COCO_V1
model = ssd300_vgg16(weights=WEIGHTS).eval()
LABELS = WEIGHTS.meta["categories"]
print(f"Loaded SSD300-VGG16 | {len(LABELS)} classes | weights: COCO_V1")

In [ ]:
from torchvision.transforms.functional import to_tensor
from tqdm import tqdm

CONF_THRESHOLD = 0.40   # discard predictions below this confidence

# ── build list of image paths from the sampled metadata ───────────────────────
img_paths = [IMG_DIR / m["file_name"] for m in sampled if (IMG_DIR / m["file_name"]).exists()]
print(f"Running inference on {len(img_paths)} images …")

all_predictions = {}   # file_name → {boxes, labels, scores}

model.eval()
with torch.no_grad():
    for path in tqdm(img_paths, desc="Inference"):
        img_pil = Image.open(path).convert("RGB")
        img_t   = to_tensor(img_pil).unsqueeze(0).to(DEVICE)   # [1, 3, H, W]

        output = model(img_t)[0]   # dict: boxes, labels, scores

        # filter by confidence
        keep   = output["scores"] >= CONF_THRESHOLD
        boxes  = output["boxes"][keep].cpu()
        labels = output["labels"][keep].cpu()
        scores = output["scores"][keep].cpu()

        all_predictions[path.name] = {
            "boxes":  boxes.tolist(),
            "labels": labels.tolist(),
            "scores": scores.tolist(),
            "label_names": [CATEGORIES[l] for l in labels.tolist()],
        }

print(f"Inference complete.  Predictions stored for {len(all_predictions)} images.")

# quick sanity check
sample_key = next(iter(all_predictions))
sample_pred = all_predictions[sample_key]
print(f"\nExample ({sample_key}): {len(sample_pred['boxes'])} detections")
for name, score in zip(sample_pred["label_names"], sample_pred["scores"]):
    print(f"  {name:<20s}  conf={score:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# deterministic colour per class
rng = np.random.default_rng(0)
CLASS_COLORS = {name: rng.integers(80, 230, size=3) / 255.0 for name in CATEGORIES}

def draw_predictions(ax, img_pil, pred, title=""):
    ax.imshow(img_pil)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
    for box, name, score in zip(pred["boxes"], pred["label_names"], pred["scores"]):
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        color = CLASS_COLORS[name]
        rect = patches.Rectangle((x1, y1), w, h,
                                  linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"{name} {score:.2f}",
                fontsize=7, color="white",
                bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"))

# ── pick 5 images that have at least one detection ────────────────────────────
vis_keys = [k for k, v in all_predictions.items() if len(v["boxes"]) > 0][:5]
assert len(vis_keys) == 5, "Could not find 5 images with detections — lower CONF_THRESHOLD."

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
fig.suptitle("SSD300-VGG16 — COCO 2017 val subset predictions", fontsize=13, y=1.02)

for ax, key in zip(axes, vis_keys):
    img = Image.open(IMG_DIR / key).convert("RGB")
    pred = all_predictions[key]
    draw_predictions(ax, img, pred, title=f"{key}\n({len(pred['boxes'])} detections)")

plt.tight_layout()
plt.savefig(DATA_DIR / "predictions_5images.png", bbox_inches="tight", dpi=150)
plt.show()
print("Figure saved to", DATA_DIR / "predictions_5images.png")